In [ ]:
# Dependencies for this project are listed in requirements.txt
# Install with: pip install -r requirements.txt
print("Environment ready.")

In [ ]:
# ==========================================
# REQUIRED LIBRARIES IMPORT
# ==========================================

# Pandas: Main tool for data manipulation and analysis of tabular data
import pandas as pd

# NumPy: Numerical operations and array handling
import numpy as np

# Matplotlib: Base charting library for visualizations
import matplotlib.pyplot as plt

# Seaborn: Statistical visualization library with enhanced styling
import seaborn as sns

# Re: Regular expressions for advanced text cleaning (currency symbols)
import re

# Display configuration for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries successfully loaded and environment configured.")

## STEP 1: DATA LOADING & EXPLORATION

In [ ]:
# ==========================================
# 1.1 LOAD DATASET FROM EXCEL FILE
# ==========================================

# Dataset: Amazon Sales Dataset from Kaggle
# Source: https://www.kaggle.com/datasets/karkavelrajaj/amazon-sales-dataset
# Contains: ~1,400 products from Amazon India across multiple categories

file_path = 'amazon.xlsx'

# Load Excel file - sheet name 'raw_data' contains the source data
df = pd.read_excel(file_path, sheet_name='raw_data')

print(f"Dataset loaded successfully!")
print(f"Total records: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print("\n" + "="*60 + "\n")
print("Column names:")
print(df.columns.tolist())

In [ ]:
# ==========================================
# 1.2 INITIAL DATA INSPECTION
# ==========================================

# Display first rows to understand data structure
print("First 10 rows of the dataset:")
display(df.head(10))

print("\n" + "="*60 + "\n")

# Data types and null values
print("Dataset info (data types and null counts):")
df.info()

print("\n" + "="*60 + "\n")

# Statistical summary of numerical columns
print("Statistical summary:")
display(df.describe())

print("\n" + "="*60 + "\n")
print("✓ Initial exploration completed.")

## STEP 2: DATA CLEANING

In [ ]:
# ==========================================
# 2.1 CLEAN PRICE COLUMNS
# ==========================================

# Problem: Prices are stored as strings with currency symbols (₹) and special characters
# Solution: Use regular expressions to extract numeric values

def clean_currency(value):
    """
    Remove currency symbols and special characters from price strings.
    Convert to float for numerical operations.
    
    Example: '₹1,299' -> 1299.0
    """
    if pd.isna(value):  # Handle missing values
        return np.nan
    
    # Convert to string and remove all non-numeric characters except decimal point
    cleaned = re.sub(r'[^0-9.]', '', str(value))
    
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

# Apply cleaning function to price columns
df['discounted_price_clean'] = df['discounted_price'].apply(clean_currency)
df['actual_price_clean'] = df['actual_price'].apply(clean_currency)

print("Price columns cleaned successfully!")
print("\nBefore cleaning (sample):")
print(df[['discounted_price', 'actual_price']].head(3))
print("\nAfter cleaning:")
print(df[['discounted_price_clean', 'actual_price_clean']].head(3))

print("\n" + "="*60 + "\n")
print("✓ Price cleaning completed.")

In [ ]:
# ==========================================
# 2.2 CLEAN RATING COLUMN
# ==========================================

# Problem: Rating column may contain non-numeric values
# Solution: Convert to numeric, replace errors with NaN, then impute with mean

# Convert to numeric (coerce errors to NaN)
df['rating_clean'] = pd.to_numeric(df['rating'], errors='coerce')

# Count missing values
missing_ratings = df['rating_clean'].isnull().sum()
print(f"Missing ratings: {missing_ratings}")

if missing_ratings > 0:
    # Calculate mean rating for imputation
    mean_rating = df['rating_clean'].mean()
    print(f"Mean rating (for imputation): {mean_rating:.2f}")
    
    # Fill missing values with mean
    df['rating_clean'] = df['rating_clean'].fillna(mean_rating)
    print(f"✓ Filled {missing_ratings} missing ratings with mean value.")
else:
    print("✓ No missing ratings found.")

print("\n" + "="*60 + "\n")
print("Rating distribution after cleaning:")
print(df['rating_clean'].describe())

print("\n" + "="*60 + "\n")
print("✓ Rating cleaning completed.")

In [ ]:
# ==========================================
# 2.3 SIMPLIFY CATEGORY COLUMN
# ==========================================

# Problem: Category column contains hierarchical paths (e.g., "Electronics|Accessories|Cables")
# Solution: Extract only the main (first-level) category for high-level analysis

# Extract main category (text before first '|')
df['main_category'] = df['category'].str.split('|').str[0]

print("Main categories extracted:")
print(df['main_category'].value_counts())

print("\n" + "="*60 + "\n")
print("✓ Category simplification completed.")

## STEP 3: FEATURE ENGINEERING (BUSINESS METRICS)

In [ ]:
# ==========================================
# 3.1 CALCULATE BUSINESS METRICS
# ==========================================

# 1. Discount Amount (Absolute savings in currency)
df['discount_amount'] = df['actual_price_clean'] - df['discounted_price_clean']

# 2. Real Discount Percentage (Recalculated from actual prices)
# Formula: ((actual - discounted) / actual) * 100
df['discount_percent_real'] = ((df['actual_price_clean'] - df['discounted_price_clean']) / df['actual_price_clean']) * 100

# 3. Star Products (High rating + High discount)
# Criteria: Rating >= 4.5 AND Discount >= 50%
df['is_star_product'] = (df['rating_clean'] >= 4.5) & (df['discount_percent_real'] >= 50)

print("Business metrics calculated:")
print(f"  - Discount Amount: Absolute savings per product")
print(f"  - Real Discount %: Recalculated discount percentage")
print(f"  - Star Products: {df['is_star_product'].sum()} products qualify (rating ≥ 4.5, discount ≥ 50%)")

print("\n" + "="*60 + "\n")

# Display sample of calculated metrics
print("Sample of business metrics:")
display(df[['product_name', 'actual_price_clean', 'discounted_price_clean', 
            'discount_amount', 'discount_percent_real', 'rating_clean', 'is_star_product']].head(10))

print("\n" + "="*60 + "\n")
print("✓ Feature engineering completed.")

## STEP 4: EXPLORATORY DATA ANALYSIS

In [ ]:
# ==========================================
# 4.1 PRICE ANALYSIS
# ==========================================

print("PRICE STATISTICS:")
print("\nDiscounted Prices:")
print(f"  Average: ₹{df['discounted_price_clean'].mean():,.2f}")
print(f"  Median: ₹{df['discounted_price_clean'].median():,.2f}")
print(f"  Min: ₹{df['discounted_price_clean'].min():,.2f}")
print(f"  Max: ₹{df['discounted_price_clean'].max():,.2f}")
print(f"  Std: ₹{df['discounted_price_clean'].std():,.2f}")

print("\nDiscount Percentage:")
print(f"  Average: {df['discount_percent_real'].mean():.1f}%")
print(f"  Median: {df['discount_percent_real'].median():.1f}%")
print(f"  Min: {df['discount_percent_real'].min():.1f}%")
print(f"  Max: {df['discount_percent_real'].max():.1f}%")

print("\n" + "="*60 + "\n")
print("✓ Price analysis completed.")

In [ ]:
# ==========================================
# 4.2 RATING ANALYSIS
# ==========================================

print("RATING STATISTICS:")
print(f"  Average rating: {df['rating_clean'].mean():.2f} / 5.0")
print(f"  Median rating: {df['rating_clean'].median():.2f}")
print(f"  Std deviation: {df['rating_clean'].std():.2f}")

print("\nRating distribution:")
rating_bins = [0, 2, 3, 4, 4.5, 5.1]
rating_labels = ['Poor (0-2)', 'Fair (2-3)', 'Good (3-4)', 'Very Good (4-4.5)', 'Excellent (4.5-5)']
df['rating_category'] = pd.cut(df['rating_clean'], bins=rating_bins, labels=rating_labels)
print(df['rating_category'].value_counts().sort_index())

print("\n" + "="*60 + "\n")
print("✓ Rating analysis completed.")

In [ ]:
# ==========================================
# 4.3 CATEGORY ANALYSIS
# ==========================================

print("CATEGORY INSIGHTS:")
print("\nTop 10 categories by product count:")
top_categories = df['main_category'].value_counts().head(10)
for category, count in top_categories.items():
    percentage = (count / len(df)) * 100
    print(f"  {category}: {count} products ({percentage:.1f}%)")

print("\n" + "="*60 + "\n")

# Average discount by category
print("Average discount by category (Top 10):")
category_discount = df.groupby('main_category')['discount_percent_real'].mean().sort_values(ascending=False).head(10)
for category, discount in category_discount.items():
    print(f"  {category}: {discount:.1f}%")

print("\n" + "="*60 + "\n")
print("✓ Category analysis completed.")

In [ ]:
# ==========================================
# 4.4 CORRELATION ANALYSIS
# ==========================================

# Analyze relationship between discount and rating
correlation = df[['discount_percent_real', 'rating_clean']].corr().iloc[0, 1]

print("DISCOUNT vs RATING CORRELATION:")
print(f"  Pearson correlation coefficient: {correlation:.3f}")
print("\nInterpretation:")
if abs(correlation) < 0.3:
    print("  → Weak correlation: Discount percentage has minimal impact on ratings.")
    print("  → Higher discounts DO NOT guarantee better customer satisfaction.")
elif correlation > 0.3:
    print("  → Positive correlation: Higher discounts tend to correlate with better ratings.")
else:
    print("  → Negative correlation: Higher discounts correlate with lower ratings.")

print("\n" + "="*60 + "\n")
print("✓ Correlation analysis completed.")

## STEP 5: DATA VISUALIZATION

In [ ]:
# ==========================================
# 5.1 CREATE VISUALIZATIONS
# ==========================================

# Set seaborn style
sns.set_theme(style="whitegrid")

# Create figure with 6 subplots
fig, axes = plt.subplots(3, 2, figsize=(15, 15))

# 1. Rating Distribution (Histogram)
axes[0, 0].hist(df['rating_clean'], bins=30, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Rating Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Rating (out of 5)')
axes[0, 0].set_ylabel('Number of Products')
axes[0, 0].axvline(df['rating_clean'].mean(), color='red', linestyle='--', label=f"Mean: {df['rating_clean'].mean():.2f}")
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Top 10 Categories (Bar Chart)
top_10_categories = df['main_category'].value_counts().head(10)
axes[0, 1].barh(top_10_categories.index, top_10_categories.values, color='salmon', edgecolor='black')
axes[0, 1].set_title('Top 10 Product Categories', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Number of Products')
axes[0, 1].set_ylabel('Category')
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. Discount Distribution (Histogram)
axes[1, 0].hist(df['discount_percent_real'], bins=30, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Discount Percentage Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Discount (%)')
axes[1, 0].set_ylabel('Number of Products')
axes[1, 0].axvline(df['discount_percent_real'].mean(), color='red', linestyle='--', label=f"Mean: {df['discount_percent_real'].mean():.1f}%")
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Discount vs Rating (Scatter Plot)
axes[1, 1].scatter(df['discount_percent_real'], df['rating_clean'], alpha=0.4, color='purple', edgecolor='black', s=30)
axes[1, 1].set_title('Discount vs Rating', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Discount (%)')
axes[1, 1].set_ylabel('Rating (out of 5)')
axes[1, 1].grid(alpha=0.3)

# 5. Price Distribution (Box Plot)
axes[2, 0].boxplot([df['discounted_price_clean']], labels=['Discounted Price'])
axes[2, 0].set_title('Price Distribution (Box Plot)', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('Price (₹)')
axes[2, 0].grid(axis='y', alpha=0.3)

# 6. Average Discount by Category (Top 10)
avg_discount_by_cat = df.groupby('main_category')['discount_percent_real'].mean().sort_values(ascending=False).head(10)
axes[2, 1].barh(avg_discount_by_cat.index, avg_discount_by_cat.values, color='orange', edgecolor='black')
axes[2, 1].set_title('Average Discount by Category (Top 10)', fontsize=12, fontweight='bold')
axes[2, 1].set_xlabel('Average Discount (%)')
axes[2, 1].set_ylabel('Category')
axes[2, 1].grid(axis='x', alpha=0.3)

# Adjust layout
plt.tight_layout()
plt.show()

print("\n" + "="*60 + "\n")
print("✓ Visualizations created successfully.")

## STEP 6: EXPORT & SUMMARY

In [ ]:
# ==========================================
# 6.1 EXPORT CLEANED DATA TO CSV
# ==========================================

# Select relevant columns for export
export_columns = [
    'product_name',
    'main_category',
    'actual_price_clean',
    'discounted_price_clean',
    'discount_amount',
    'discount_percent_real',
    'rating_clean',
    'is_star_product'
]

# Create export dataframe
df_export = df[export_columns].copy()

# Rename columns for clarity
df_export.columns = [
    'Product Name',
    'Category',
    'Original Price (₹)',
    'Discounted Price (₹)',
    'Discount Amount (₹)',
    'Discount (%)',
    'Rating',
    'Star Product'
]

# Export to CSV
output_filename = 'amazon_sales_cleaned.csv'
df_export.to_csv(output_filename, index=False, encoding='utf-8')

print(f"✓ Data exported successfully to: {output_filename}")
print(f"Total records exported: {len(df_export)}")
print(f"Total columns: {len(df_export.columns)}")

print("\n" + "="*60 + "\n")

# Display preview of exported data
print("Preview of exported data:")
display(df_export.head(10))

print("\n" + "="*60 + "\n")
print("✓ Export completed successfully.")

In [ ]:
# ==========================================
# 6.2 PROJECT SUMMARY AND INSIGHTS
# ==========================================

print("="*60)
print("AMAZON SALES EDA - EXECUTIVE SUMMARY")
print("="*60)
print()

# Dataset overview
print("DATASET OVERVIEW:")
print(f"  Source: Amazon India Sales Dataset (Kaggle)")
print(f"  Total products analyzed: {len(df):,}")
print(f"  Product categories: {df['main_category'].nunique()}")
print()

# Price insights
print("PRICE ANALYSIS:")
print(f"  Average discounted price: ₹{df['discounted_price_clean'].mean():,.2f}")
print(f"  Median discounted price: ₹{df['discounted_price_clean'].median():,.2f}")
print(f"  Price range: ₹{df['discounted_price_clean'].min():,.2f} - ₹{df['discounted_price_clean'].max():,.2f}")
print()

# Discount insights
print("DISCOUNT ANALYSIS:")
print(f"  Average discount: {df['discount_percent_real'].mean():.1f}%")
print(f"  Maximum discount: {df['discount_percent_real'].max():.1f}%")
print(f"  Products with >50% discount: {(df['discount_percent_real'] > 50).sum()} ({(df['discount_percent_real'] > 50).sum() / len(df) * 100:.1f}%)")
print()

# Rating insights
print("RATING ANALYSIS:")
print(f"  Average rating: {df['rating_clean'].mean():.2f} / 5.0")
print(f"  Products with rating ≥ 4.5: {(df['rating_clean'] >= 4.5).sum()} ({(df['rating_clean'] >= 4.5).sum() / len(df) * 100:.1f}%)")
print()

# Star products
star_count = df['is_star_product'].sum()
print("STAR PRODUCTS (Rating ≥ 4.5 AND Discount ≥ 50%):")
print(f"  Total star products: {star_count} ({star_count / len(df) * 100:.1f}% of catalog)")
print()

# Category insights
print("TOP 3 CATEGORIES:")
top_3_categories = df['main_category'].value_counts().head(3)
for i, (category, count) in enumerate(top_3_categories.items(), 1):
    print(f"  {i}. {category}: {count} products ({count / len(df) * 100:.1f}%)")
print()

# Correlation
correlation = df[['discount_percent_real', 'rating_clean']].corr().iloc[0, 1]
print("DISCOUNT vs RATING:")
print(f"  Correlation coefficient: {correlation:.3f}")
if abs(correlation) < 0.3:
    print("  → Weak correlation: Higher discounts do NOT guarantee better ratings")
print()

print("="*60)
print("KEY FINDINGS:")
print("="*60)
print()
print("1. AGGRESSIVE DISCOUNTING STRATEGY")
print(f"   → Average discount of {df['discount_percent_real'].mean():.1f}% indicates highly competitive pricing")
print(f"   → Over half of products ({(df['discount_percent_real'] > 50).sum() / len(df) * 100:.1f}%) have discounts exceeding 50%")
print()
print("2. STRONG CUSTOMER SATISFACTION")
print(f"   → Average rating of {df['rating_clean'].mean():.2f}/5.0 reflects good product quality")
print(f"   → {(df['rating_clean'] >= 4.5).sum() / len(df) * 100:.1f}% of products have excellent ratings (≥4.5)")
print()
print("3. DISCOUNT-RATING INDEPENDENCE")
print(f"   → Correlation of {correlation:.3f} shows discount % has minimal impact on ratings")
print("   → Price cuts alone don't improve customer satisfaction")
print()
print("4. CATEGORY CONCENTRATION")
top_cat = df['main_category'].value_counts().index[0]
top_cat_pct = (df['main_category'].value_counts().iloc[0] / len(df)) * 100
print(f"   → '{top_cat}' dominates with {top_cat_pct:.1f}% of total products")
print("   → Technology categories show highest product density")
print()

print("="*60)
print("✓ PROJECT COMPLETED SUCCESSFULLY")
print("="*60)
print()
print(f"Results exported to: {output_filename}")
print("Use this file for further analysis, dashboards, or business insights.")